In [ ]:
import pandas as pd
import numpy as np
import os

# Define path to the WELFake dataset file
dataset_path = os.path.join("..", "data", "WELFake_Dataset.csv")

print("Libraries imported successfully!")

In [2]:
%pip install scikit-learn pandas numpy

Defaulting to user installation because normal site-packages is not writeable
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 645.7 kB/s eta 0:00:12
   --- ------------------------------------ 0.8/8.2 MB 762.0 kB/s eta 0:00:10
   --- ------------------------------------ 0.8/8.2 MB 762.0 kB/s eta 0:00:10
   ----- ---------------------------------- 1.0/8.2 MB 699.0 kB/s eta 0:00:11
   ------ --------------------------------- 1.3/8.2 MB 808.5 kB/s et


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

print("All advanced modeling packages loaded successfully!")

All advanced modeling packages loaded successfully!


In [6]:
import os
print(os.listdir("../data/"))

['.gitkeep', 'notebooks']


In [8]:
import os

print("--- Current Location ---")
print(os.getcwd())

print("\n--- Files in the Data Folder ---")
try:
    print(os.listdir("../data"))
except Exception as e:
    print(f"Error accessing folder: {e}")

--- Current Location ---
c:\Users\chamo\OneDrive\Desktop\github\NLP_Group_09\notebooks

--- Files in the Data Folder ---
['.gitkeep', 'notebooks']


In [11]:
import os
print(os.listdir(r"C:\Users\chamo\OneDrive\Desktop\github\NLP_Group_09\data"))

['.gitkeep', 'notebooks']


In [13]:
import os

# This forces Python to look directly at your actual Windows folder path
absolute_path = r"C:\Users\chamo\OneDrive\Desktop\github\NLP_Group_09\data\WELFake_Dataset.csv"

# Load the dataset
df = pd.read_csv(absolute_path)
df = df.dropna()
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (71537, 4)


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
5,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1


In [14]:
def clean_text(text):
    text = str(text).lower() # Convert to lowercase
    text = re.sub(r'\[.*?\]', '', text) # Remove text in brackets
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'<.*?>+', '', text) # Remove HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text) # Remove punctuation
    text = re.sub(r'\n', '', text) # Remove newlines
    text = re.sub(r'\w*\d\w*', '', text) # Remove words containing numbers
    return text

# Apply the cleaning to the text columns (using a sample to test it fast)
# WELFake has 'title' and 'text' columns. Let's create a combined column:
df['total_content'] = df['title'] + " " + df['text']

print("Cleaning a small sample of the dataset...")
df['cleaned_content'] = df['total_content'].head(1000).apply(clean_text)
print("Done sample cleaning!")

Cleaning a small sample of the dataset...
Done sample cleaning!


In [16]:
# 1. Fill any accidental missing values with an empty string to keep data safe
df['cleaned_content'] = df['cleaned_content'].fillna("")

# 2. Filter out any rows that are completely empty strings
df = df[df['cleaned_content'].str.strip() != ""]

# 3. Define features (X) and target labels (y)
X = df['cleaned_content']
y = df['label']

# 4. Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# 6. Fit and transform the text data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Training features shape: {X_train_tfidf.shape}")
print(f"Testing features shape: {X_test_tfidf.shape}")

Training features shape: (799, 5000)
Testing features shape: (200, 5000)


In [17]:
# Initialize the Linear Support Vector Classifier
svm_model = LinearSVC(random_state=42, max_iter=2000)

print("Training the SVM model (this may take a few moments)...")
svm_model.fit(X_train_tfidf, y_train)
print("Model training complete!")

Training the SVM model (this may take a few moments)...
Model training complete!


In [18]:
# Make predictions on the test set
y_pred = svm_model.predict(X_test_tfidf)

# Calculate accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"--- SVM Model Accuracy: {accuracy * 100:.2f}% ---\n")

# Print complete classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))

--- SVM Model Accuracy: 90.50% ---

Classification Report:
              precision    recall  f1-score   support

    Real (0)       0.86      0.95      0.90        93
    Fake (1)       0.95      0.87      0.91       107

    accuracy                           0.91       200
   macro avg       0.91      0.91      0.90       200
weighted avg       0.91      0.91      0.91       200

